# CEFR 3-Band Classification via a 0-100 Score - 10 Methods

**Business pipeline (fixed):** every method must produce a **0-100 score**, then **2 cut-points**
on that score split it into the **3 bands**.

```
features -> model -> class probabilities -> 0-100 score -> 2 cut-points -> 3 bands
```

**Bands (confirmed with senior):**

| Band | CEFR levels | Code |
|---|---|---|
| 0 | A1, A2 | `A1-A2` |
| 1 | B1 | `B1` |
| 2 | B2, C1, C2 | `B2-C1-C2` |

**Baseline to beat:** 77% accuracy on these 3 bands (senior's regression models). **Target:** >=82%.

**Notes**
- `X_train / X_test` are built **once** from the `split` column and **shared by all 10 methods**.
- Cut-points are tuned on **train only**, then applied unchanged to test (no leakage).
- Each method also reports its plain `argmax` accuracy so you can see whether the
  0-100 score + tuned cut-points actually *helps* (the crux test).

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    cohen_kappa_score, confusion_matrix, classification_report,
)

# ---- optional libraries (each method degrades gracefully if missing) ----
try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    from interpret.glassbox import ExplainableBoostingClassifier
    HAS_EBM = True
except Exception:
    HAS_EBM = False

try:
    import mord
    HAS_MORD = True
except Exception:
    HAS_MORD = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

print("lightgbm:", HAS_LGBM, "| interpret(EBM):", HAS_EBM, "| mord:", HAS_MORD, "| shap:", HAS_SHAP)
print("Install any missing:  pip install lightgbm interpret mord shap")

## 1. Load your data  <-- FILL THIS IN

Assign your dataframe to `df`. Nothing else in the notebook needs changing here.

In [ ]:
# TODO: initialise your dataframe here
df = None

# e.g. df = pd.read_csv("your_file.csv")
# e.g. df = pd.read_excel("your_file.xlsx")

## 2. Columns  <-- FILL THIS IN

Put your chosen features (one per feature group, 8-11 total) in `FEATURE_COLS`.
Everything else is metadata: `ciid`, `location`, `split`, and the label.

In [ ]:
# TODO: one feature per feature group (8-11 total)
FEATURE_COLS = []

# ---- metadata columns (edit names if yours differ) ----
ID_COL       = "ciid"
LOCATION_COL = "location"
SPLIT_COL    = "split"
LABEL_COL    = "cefr"       # TODO: your CEFR label column

# values inside SPLIT_COL that define the sets; anything else is dropped (bad flags)
TRAIN_VALUE = "train"
TEST_VALUE  = "test"

META_COLS = [ID_COL, LOCATION_COL, SPLIT_COL, LABEL_COL]

## 3. Configuration

In [ ]:
RANDOM_STATE = 42

# Band definition: A1,A2 -> 0 | B1 -> 1 | B2,C1,C2 -> 2
BAND_MAP = {
    "A1": 0, "A2": 0,
    "B1": 1,
    "B2": 2, "C1": 2, "C2": 2,
}
BAND_NAMES = ["A1-A2", "B1", "B2-C1-C2"]
N_BANDS = 3

# Score anchors: score = sum_k P(band_k) * anchor_k  -> lands in [0, 100].
# NOTE: because the 2 cut-points are *tuned*, any monotonic anchor set gives the
# same accuracy. Anchors only change how the 0-100 number *reads* to the business.
BAND_ANCHORS = np.array([0.0, 50.0, 100.0])      # plain expected-index scaling
# BAND_ANCHORS = np.array([29.0, 50.0, 74.0])    # CEFR/GSE-flavoured alternative

# metric the cut-points are optimised for on TRAIN
OPTIMIZE_METRIC = "accuracy"     # or "balanced_accuracy"

BASELINE_ACC = 0.77              # senior's regression baseline
TARGET_ACC   = 0.82              # goal

## 4. Build train / test  (shared by all 10 methods)

Rows whose `split` is neither `train` nor `test` (the "bad flags") are dropped.

In [ ]:
assert df is not None, "Assign your dataframe to `df` in section 1."
assert len(FEATURE_COLS) > 0, "Fill FEATURE_COLS in section 2."

missing = [c for c in FEATURE_COLS + META_COLS if c not in df.columns]
assert not missing, f"Columns not found in df: {missing}"


def to_band(s):
    """Map CEFR labels to band 0/1/2. Accepts 6-level strings or already-coded 0/1/2."""
    s = pd.Series(s)
    if s.dtype.kind in "iuf":
        vals = set(pd.unique(s.dropna()))
        if vals <= {0, 1, 2}:
            return s.astype(int).to_numpy()
    key = s.astype(str).str.strip().str.upper().str.replace(" ", "", regex=False)
    mapped = key.map(BAND_MAP)
    bad = key[mapped.isna()].unique()
    assert len(bad) == 0, f"Unmapped label values: {bad}"
    return mapped.astype(int).to_numpy()


split_norm = df[SPLIT_COL].astype(str).str.strip().str.lower()
train_mask = split_norm == TRAIN_VALUE
test_mask  = split_norm == TEST_VALUE
dropped    = (~train_mask & ~test_mask).sum()

train_df = df.loc[train_mask].copy()
test_df  = df.loc[test_mask].copy()

X_train = train_df[FEATURE_COLS].astype(float)
X_test  = test_df[FEATURE_COLS].astype(float)
y_train = to_band(train_df[LABEL_COL])
y_test  = to_band(test_df[LABEL_COL])

print(f"features           : {len(FEATURE_COLS)}  -> {FEATURE_COLS}")
print(f"train / test rows  : {len(X_train)} / {len(X_test)}")
print(f"dropped (bad flags): {dropped}")
print(f"other split values : {sorted(set(split_norm) - {TRAIN_VALUE, TEST_VALUE})}")
print()
print("train band distribution:")
print(pd.Series(y_train).value_counts().sort_index()
        .rename(lambda i: f"{i} = {BAND_NAMES[i]}"))
print()
print("test band distribution:")
print(pd.Series(y_test).value_counts().sort_index()
        .rename(lambda i: f"{i} = {BAND_NAMES[i]}"))

## 5. Core utilities

- `FrankHallOrdinal` - ordinal decomposition (Frank & Hall 2001): trains `K-1` binary
  classifiers for `P(y > k)` and differences them into per-band probabilities.
- `proba_to_score` - probabilities -> the **0-100 score**.
- `fit_cutpoints` - grid-search the **2 cut-points** on train.
- `evaluate` - runs the full pipeline for one method and records the result.

In [ ]:
class FrankHallOrdinal(BaseEstimator, ClassifierMixin):
    """Ordinal decomposition (Frank & Hall, 2001).

    For K ordered classes, fit K-1 binary models P(y > k), then difference the
    cumulative probabilities into per-class probabilities.
    """

    def __init__(self, base_estimator=None):
        self.base_estimator = base_estimator

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.K_ = len(self.classes_)
        self.models_ = []
        for k in range(self.K_ - 1):
            yk = (y > self.classes_[k]).astype(int)
            if len(np.unique(yk)) < 2:            # degenerate split
                self.models_.append(("const", float(yk[0])))
            else:
                m = clone(self.base_estimator)
                m.fit(X, yk)
                self.models_.append(("model", m))
        return self

    def predict_proba(self, X):
        n = len(X)
        cum = np.zeros((n, self.K_ - 1))
        for k, (kind, m) in enumerate(self.models_):
            cum[:, k] = m if kind == "const" else m.predict_proba(X)[:, 1]
        # cumulative P(y>k) must be non-increasing in k
        cum = np.minimum.accumulate(cum, axis=1)

        p = np.zeros((n, self.K_))
        p[:, 0] = 1.0 - cum[:, 0]
        for k in range(1, self.K_ - 1):
            p[:, k] = cum[:, k - 1] - cum[:, k]
        p[:, -1] = cum[:, -1]

        p = np.clip(p, 1e-9, None)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


def make_pipe(estimator, scale=True):
    """Impute -> (optionally) standardise -> estimator."""
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)


def proba_to_score(proba, anchors=BAND_ANCHORS):
    """Per-band probabilities -> single 0-100 score."""
    return np.asarray(proba) @ np.asarray(anchors, dtype=float)


def apply_cutpoints(scores, t1, t2):
    """2 cut-points on the 0-100 score -> 3 bands."""
    scores = np.asarray(scores)
    return np.where(scores <= t1, 0, np.where(scores <= t2, 1, 2))


def _scorer(name):
    return accuracy_score if name == "accuracy" else balanced_accuracy_score


def fit_cutpoints(scores, y, metric=OPTIMIZE_METRIC, n_grid=120):
    """Grid-search the 2 thresholds that maximise `metric` on the given data."""
    scores = np.asarray(scores)
    sc = _scorer(metric)
    cand = np.unique(np.percentile(scores, np.linspace(0, 100, n_grid)))
    best_val, best_cuts = -1.0, (33.3, 66.7)
    for i in range(len(cand) - 1):
        for j in range(i + 1, len(cand)):
            pred = apply_cutpoints(scores, cand[i], cand[j])
            v = sc(y, pred)
            if v > best_val:
                best_val, best_cuts = v, (float(cand[i]), float(cand[j]))
    return best_cuts, best_val


CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

RESULTS = []
FITTED = {}


def _metrics(y, pred):
    return dict(
        acc=accuracy_score(y, pred),
        bal_acc=balanced_accuracy_score(y, pred),
        macro_f1=f1_score(y, pred, average="macro"),
        qwk=cohen_kappa_score(y, pred, weights="quadratic"),
        mae=float(np.mean(np.abs(np.asarray(y) - np.asarray(pred)))),
    )


def evaluate(name, model, anchors=BAND_ANCHORS, verbose=True):
    """OOF probabilities -> 0-100 score -> tune 2 cut-points -> apply to test.

    Cut-points are tuned on *out-of-fold* train probabilities, never in-sample.
    In-sample probabilities are optimistically biased for every model, and are
    outright degenerate for kNN with distance weights (each training point is its
    own zero-distance neighbour => probability 1.0 on its true class). Cut-points
    tuned on such leaked scores do not transfer to test.
    """
    # --- out-of-fold train probabilities -> honest cut-points ---
    p_oof = cross_val_predict(clone(model), X_train, y_train,
                              cv=CV, method="predict_proba")
    s_oof = proba_to_score(p_oof, anchors)
    (t1, t2), _ = fit_cutpoints(s_oof, y_train)

    # --- refit on all of train, then score train (in-sample) and test ---
    model.fit(X_train, y_train)
    p_tr = model.predict_proba(X_train)          # in-sample
    p_te = model.predict_proba(X_test)
    s_tr, s_te = proba_to_score(p_tr, anchors), proba_to_score(p_te, anchors)

    pred_oof = apply_cutpoints(s_oof, t1, t2)
    pred_tr = apply_cutpoints(s_tr, t1, t2)      # in-sample train
    pred_te = apply_cutpoints(s_te, t1, t2)

    # whole dataset = train (in-sample) + test (held out)
    y_full = np.concatenate([y_train, y_test])
    pred_full = np.concatenate([pred_tr, pred_te])

    m_te = _metrics(y_test, pred_te)
    argmax_acc = accuracy_score(y_test, np.argmax(p_te, axis=1))

    row = dict(method=name,
               train_acc=accuracy_score(y_train, pred_tr),
               test_acc=m_te["acc"],
               full_acc=accuracy_score(y_full, pred_full),
               oof_acc=accuracy_score(y_train, pred_oof),
               test_bal_acc=m_te["bal_acc"], macro_f1=m_te["macro_f1"],
               qwk=m_te["qwk"], mae=m_te["mae"],
               argmax_acc=argmax_acc, lift_vs_argmax=m_te["acc"] - argmax_acc,
               cut1=t1, cut2=t2)
    RESULTS.append(row)
    FITTED[name] = dict(model=model, score_test=s_te, pred_test=pred_te,
                        score_train=s_tr, pred_train=pred_tr,
                        proba_test=p_te, proba_train=p_tr, proba_oof=p_oof,
                        cuts=(t1, t2))

    if verbose:
        flag = "PASS" if m_te["acc"] >= TARGET_ACC else ("ok" if m_te["acc"] >= BASELINE_ACC else "-")
        print(f"{name}")
        print(f"  TRAIN {row['train_acc']:.3f} | TEST {m_te['acc']:.3f} | "
              f"FULL {row['full_acc']:.3f} | OOF {row['oof_acc']:.3f}   [{flag}]")
        print(f"  bal {m_te['bal_acc']:.3f} | macroF1 {m_te['macro_f1']:.3f} | "
              f"QWK {m_te['qwk']:.3f}")
        print(f"  cut-points ({t1:.1f}, {t2:.1f}) | argmax {argmax_acc:.3f} "
              f"| lift {m_te['acc'] - argmax_acc:+.3f}")
    return row


def evaluate_scores(name, s_oof, s_te, s_tr_in=None, verbose=True):
    """As `evaluate`, for methods that emit a 0-100 score directly.

    `s_oof`   - out-of-fold (or label-free, e.g. PCA) train scores; cut-points are fit on these.
    `s_tr_in` - in-sample train scores for the TRAIN accuracy column. Defaults to `s_oof`
                (correct for label-free methods such as PCA, where the two are identical).
    """
    if s_tr_in is None:
        s_tr_in = s_oof

    (t1, t2), _ = fit_cutpoints(s_oof, y_train)
    pred_oof = apply_cutpoints(s_oof, t1, t2)
    pred_tr = apply_cutpoints(s_tr_in, t1, t2)
    pred_te = apply_cutpoints(s_te, t1, t2)

    y_full = np.concatenate([y_train, y_test])
    pred_full = np.concatenate([pred_tr, pred_te])
    m_te = _metrics(y_test, pred_te)

    row = dict(method=name,
               train_acc=accuracy_score(y_train, pred_tr),
               test_acc=m_te["acc"],
               full_acc=accuracy_score(y_full, pred_full),
               oof_acc=accuracy_score(y_train, pred_oof),
               test_bal_acc=m_te["bal_acc"], macro_f1=m_te["macro_f1"],
               qwk=m_te["qwk"], mae=m_te["mae"],
               argmax_acc=np.nan, lift_vs_argmax=np.nan, cut1=t1, cut2=t2)
    RESULTS.append(row)
    FITTED[name] = dict(model=None, score_test=s_te, pred_test=pred_te,
                        score_train=s_tr_in, pred_train=pred_tr,
                        proba_test=None, proba_train=None, proba_oof=None,
                        cuts=(t1, t2))

    if verbose:
        flag = "PASS" if m_te["acc"] >= TARGET_ACC else ("ok" if m_te["acc"] >= BASELINE_ACC else "-")
        print(f"{name}")
        print(f"  TRAIN {row['train_acc']:.3f} | TEST {m_te['acc']:.3f} | "
              f"FULL {row['full_acc']:.3f} | OOF {row['oof_acc']:.3f}   [{flag}]")
        print(f"  bal {m_te['bal_acc']:.3f} | macroF1 {m_te['macro_f1']:.3f} | "
              f"QWK {m_te['qwk']:.3f}")
        print(f"  cut-points ({t1:.1f}, {t2:.1f})")
    return row


print("utilities ready")

# Methods

Ten approaches, one per cell, ranked by expected chance of clearing 82%.
All share the same `X_train / X_test`, and all end in the same
**0-100 score -> 2 cut-points -> 3 bands** pipeline.

## Method 1 - Ordinal Gradient Boosting (LightGBM / sklearn GBM + Frank-Hall)

Highest accuracy ceiling. Shallow trees + strong regularisation to survive n~220.
Falls back to sklearn's `GradientBoostingClassifier` if LightGBM is unavailable.

In [ ]:
if HAS_LGBM:
    base = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.03, max_depth=3, num_leaves=7,
        min_child_samples=15, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8, reg_lambda=1.0, random_state=RANDOM_STATE, verbose=-1,
    )
else:
    base = GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.03, max_depth=2,
        min_samples_leaf=15, subsample=0.8, random_state=RANDOM_STATE,
    )

evaluate("1. Ordinal GBM (Frank-Hall)",
         make_pipe(FrankHallOrdinal(base), scale=False))

## Method 2 - Explainable Boosting Machine (EBM / GA2M)

Glass-box additive model: each feature has an exact additive contribution, so the
per-section explanation is native rather than post-hoc.

In [ ]:
if HAS_EBM:
    ebm = ExplainableBoostingClassifier(
        interactions=5, outer_bags=8, random_state=RANDOM_STATE,
    )
    evaluate("2. EBM (GA2M)", make_pipe(ebm, scale=False))
else:
    print("skipped - `pip install interpret` to enable EBM")

## Method 3 - SVM (RBF) + Frank-Hall + Platt calibration

Classic small-n champion and the historical CEFR workhorse. Scaling is essential.

In [ ]:
svc = SVC(kernel="rbf", C=3.0, gamma="scale",
          probability=True, class_weight="balanced", random_state=RANDOM_STATE)
svc_cal = CalibratedClassifierCV(svc, cv=3, method="sigmoid")

evaluate("3. SVM-RBF (Frank-Hall + Platt)",
         make_pipe(FrankHallOrdinal(svc_cal), scale=True))

## Method 4 - Ordinal Random Forest (RF + Frank-Hall)

Bagging keeps variance low - often the most *stable* strong model at this sample size.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=800, max_features="sqrt", min_samples_leaf=3,
    class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1,
)

evaluate("4. Ordinal RF (Frank-Hall)",
         make_pipe(FrankHallOrdinal(rf), scale=False))

## Method 5 - LDA / QDA

The strongest of the simple methods; shrinkage LDA is very stable at small n.
Both are run and the better one is kept.

In [ ]:
evaluate("5a. LDA (shrinkage)",
         make_pipe(LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"), scale=True))

evaluate("5b. QDA",
         make_pipe(QuadraticDiscriminantAnalysis(reg_param=0.3), scale=True))

## Method 6 - Proportional-Odds Ordinal Logistic (baseline)

The honest reference line. Uses `mord.LogisticAT` when available, otherwise a
Frank-Hall logistic approximation. *This is technically "regression" - benchmark only.*

In [ ]:
if HAS_MORD:
    class MordWrapper(BaseEstimator, ClassifierMixin):
        def __init__(self, alpha=1.0):
            self.alpha = alpha
        def fit(self, X, y):
            self.m_ = mord.LogisticAT(alpha=self.alpha).fit(np.asarray(X), np.asarray(y))
            self.classes_ = np.unique(y)
            return self
        def predict_proba(self, X):
            return self.m_.predict_proba(np.asarray(X))
        def predict(self, X):
            return self.m_.predict(np.asarray(X))

    evaluate("6. Ordinal Logistic (mord, baseline)", make_pipe(MordWrapper(1.0), scale=True))
else:
    logit = LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced")
    evaluate("6. Ordinal Logistic (Frank-Hall approx, baseline)",
             make_pipe(FrankHallOrdinal(logit), scale=True))

## Method 7 - Gaussian Naive Bayes

Very low variance; capped by the (violated) feature-independence assumption.
Calibrated because raw NB probabilities are typically over-confident.

In [ ]:
nb = CalibratedClassifierCV(GaussianNB(), cv=3, method="sigmoid")
evaluate("7. Gaussian Naive Bayes", make_pipe(nb, scale=True))

## Method 8 - k-Nearest Neighbours (distance-weighted)

Case-based. Also shown: a `KNeighborsRegressor` on the band index, which yields a
smooth 0-100 score directly without any anchor mapping.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=11, weights="distance")
evaluate("8a. kNN (k=11, distance)", make_pipe(knn, scale=True))

# variant: regress the band index -> score maps straight to 0-100
from sklearn.neighbors import KNeighborsRegressor
knn_r = make_pipe(KNeighborsRegressor(n_neighbors=11, weights="distance"), scale=True)

# OOF is essential here: in-sample, every point is its own 0-distance neighbour
s_oof = cross_val_predict(clone(knn_r), X_train, y_train, cv=CV) / (N_BANDS - 1) * 100
knn_r.fit(X_train, y_train)
s_tr_in = knn_r.predict(X_train) / (N_BANDS - 1) * 100
s_te = knn_r.predict(X_test) / (N_BANDS - 1) * 100
evaluate_scores("8b. kNN regressor on band index", s_oof, s_te, s_tr_in=s_tr_in)

## Method 9 - Single Decision Tree (shallow, pruned)

Low ceiling and unstable, but produces human-readable if-then rules.

In [ ]:
dt = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=10, ccp_alpha=0.005,
    class_weight="balanced", random_state=RANDOM_STATE,
)
evaluate("9. Decision Tree (shallow)", make_pipe(dt, scale=False))

# readable rules
from sklearn.tree import export_text
_dt = make_pipe(clone(dt), scale=False).fit(X_train, y_train)
print(export_text(_dt.named_steps["model"], feature_names=list(FEATURE_COLS)))

## Method 10 - Composite Index / PCA scoring (rubric)

The simplest possible realisation of the requirement: normalise the features, combine
them into one index (PCA first component - unsupervised, so **not** regression), scale to
0-100, then cut. Lowest accuracy ceiling but maximum transparency.

In [ ]:
pca_pipe = make_pipe(PCA(n_components=1, random_state=RANDOM_STATE), scale=True)
pc_tr = pca_pipe.fit_transform(X_train).ravel()
pc_te = pca_pipe.transform(X_test).ravel()

# orient the component so higher = more proficient
if np.corrcoef(pc_tr, y_train)[0, 1] < 0:
    pc_tr, pc_te = -pc_tr, -pc_te

lo, hi = pc_tr.min(), pc_tr.max()
# PCA never sees y, so in-sample train scores are already unbiased here
s_oof = np.clip((pc_tr - lo) / (hi - lo), 0, 1) * 100
s_te = np.clip((pc_te - lo) / (hi - lo), 0, 1) * 100

evaluate_scores("10. Composite Index (PCA-1)", s_oof, s_te)

loadings = pd.Series(
    pca_pipe.named_steps["model"].components_[0], index=FEATURE_COLS
).sort_values(key=np.abs, ascending=False)
print("\nPC-1 loadings (per-section weight in the rubric):")
print(loadings.round(3))

# Results

## Leaderboard

Every method reports four accuracies:

| Column | Meaning | Honest generalisation estimate? |
|---|---|---|
| `train_acc` | in-sample: model fitted on train, predicted on train | **No** - optimistic by construction |
| `test_acc`  | held-out test set | **Yes** |
| `full_acc`  | train + test combined (whole dataset) | **No** - ~2/3 of it is the in-sample train part |
| `oof_acc`   | out-of-fold predictions on train | **Yes** - the most stable one |

**Read them this way:**
- `train_acc` and `full_acc` are **fit diagnostics, not performance claims.** Do not quote
  `full_acc` to the business as "our accuracy" - it is inflated by the in-sample train rows.
- A large `train_acc - oof_acc` gap = overfitting (expect this for RF / boosting / kNN).
- **Pick your method on `oof_acc`, then report `test_acc`.** Choosing the winner by the
  highest `test_acc` is itself overfitting to the test set - with ~70 test rows a 3-4 point
  gap is well within noise.

In [ ]:
lb = (pd.DataFrame(RESULTS)
        .sort_values("test_acc", ascending=False)
        .reset_index(drop=True))

lb["vs_baseline"] = lb["test_acc"] - BASELINE_ACC
lb["status"] = np.where(lb["test_acc"] >= TARGET_ACC, "PASS >=82%",
                 np.where(lb["test_acc"] >= BASELINE_ACC, "beats 77%", "below 77%"))

pd.set_option("display.width", 200, "display.max_columns", 50)
display(lb[["method", "train_acc", "test_acc", "full_acc", "oof_acc",
            "test_bal_acc", "macro_f1", "qwk", "mae",
            "argmax_acc", "lift_vs_argmax", "cut1", "cut2",
            "vs_baseline", "status"]].round(4))

print("\ntrain_acc = in-sample (optimistic) | test_acc = held out | "
      "full_acc = train+test combined | oof_acc = out-of-fold (honest)")
print("gap train_acc - oof_acc is your overfitting indicator:")
display((lb[["method", "train_acc", "oof_acc"]]
           .assign(overfit_gap=lb["train_acc"] - lb["oof_acc"])
           .sort_values("overfit_gap", ascending=False)).round(4))

print(f"\nbaseline {BASELINE_ACC:.0%} | target {TARGET_ACC:.0%}")
print(f"best: {lb.iloc[0]['method']}  ->  {lb.iloc[0]['test_acc']:.3f}")

## Does the 0-100 score actually beat plain `argmax`?

This is the crux test: positive `lift_vs_argmax` means the score + tuned cut-points
genuinely add classification power over reading `argmax` off the model.

In [ ]:
crux = lb.dropna(subset=["lift_vs_argmax"])[
    ["method", "test_acc", "argmax_acc", "lift_vs_argmax"]
].sort_values("lift_vs_argmax", ascending=False)
display(crux.round(4))

print(f"mean lift: {crux['lift_vs_argmax'].mean():+.4f}")
print(f"methods helped by the score: {(crux['lift_vs_argmax'] > 0).sum()} / {len(crux)}")

## Confusion matrix + report for the best method

In [ ]:
best_name = lb.iloc[0]["method"]
best = FITTED[best_name]

print(f"=== {best_name} ===  cut-points: {best['cuts'][0]:.1f}, {best['cuts'][1]:.1f}\n")
cm = confusion_matrix(y_test, best["pred_test"])
display(pd.DataFrame(cm,
                     index=[f"true {b}" for b in BAND_NAMES],
                     columns=[f"pred {b}" for b in BAND_NAMES]))
print(classification_report(y_test, best["pred_test"], target_names=BAND_NAMES))

## Interpretability - per-feature (per-section) importance

Permutation importance on the best model. Map each feature back to its feature group
to report a **section-level** contribution.

In [ ]:
if FITTED[best_name]["model"] is not None:
    pi = permutation_importance(
        FITTED[best_name]["model"], X_test, y_test,
        n_repeats=30, random_state=RANDOM_STATE, scoring="accuracy",
    )
    imp = (pd.DataFrame({"feature": FEATURE_COLS,
                         "importance": pi.importances_mean,
                         "std": pi.importances_std})
             .sort_values("importance", ascending=False)
             .reset_index(drop=True))
    display(imp.round(4))
else:
    print(f"{best_name} has no sklearn model object (score-based method).")

# EBM native per-feature contributions (exact, not post-hoc)
if HAS_EBM and "2. EBM (GA2M)" in FITTED:
    ebm_model = FITTED["2. EBM (GA2M)"]["model"].named_steps["model"]
    g = ebm_model.explain_global()
    print("\nEBM global term importances:")
    for n, v in sorted(zip(g.data()["names"], g.data()["scores"]),
                       key=lambda t: -t[1]):
        print(f"  {n:<30} {v:.4f}")

## Soft-voting ensemble of the top 3

Averaging probabilities from diverse strong models typically adds 1-3 points.

In [ ]:
# select members by OOF accuracy (not test) to avoid overfitting the test set
top3 = (pd.DataFrame(RESULTS)
          .sort_values("oof_acc", ascending=False)["method"].tolist())
top3 = [m for m in top3 if FITTED[m]["proba_oof"] is not None][:3]
print("ensembling:", top3)

if len(top3) >= 2:
    P_oof = np.mean([FITTED[m]["proba_oof"] for m in top3], axis=0)
    P_tr = np.mean([FITTED[m]["proba_train"] for m in top3], axis=0)
    P_te = np.mean([FITTED[m]["proba_test"] for m in top3], axis=0)
    s_oof_ens = proba_to_score(P_oof)
    s_tr_ens = proba_to_score(P_tr)
    s_te_ens = proba_to_score(P_te)

    evaluate_scores(f"ENSEMBLE ({len(top3)} models)", s_oof_ens, s_te_ens,
                    s_tr_in=s_tr_ens)

    lb2 = (pd.DataFrame(RESULTS).sort_values("test_acc", ascending=False)
             .reset_index(drop=True))
    display(lb2[["method", "test_acc", "test_bal_acc", "qwk"]].round(4).head(10))

## Final 0-100 scores for the test set

The business deliverable: a per-row 0-100 score plus the band it falls into.

In [ ]:
out = pd.DataFrame({
    ID_COL:       test_df[ID_COL].values,
    LOCATION_COL: test_df[LOCATION_COL].values,
    "score_0_100": np.round(FITTED[best_name]["score_test"], 2),
    "pred_band":   [BAND_NAMES[i] for i in FITTED[best_name]["pred_test"]],
    "true_band":   [BAND_NAMES[i] for i in y_test],
})
out["correct"] = out["pred_band"] == out["true_band"]
display(out.head(20))

print(f"\nmethod: {best_name}")
print(f"cut-points: {FITTED[best_name]['cuts'][0]:.1f} / {FITTED[best_name]['cuts'][1]:.1f}")
print(f"accuracy: {out['correct'].mean():.3f}")

# out.to_csv("test_scores.csv", index=False)